> **Public reproducibility notebook.** This notebook contains the experiment logic used in the paper. Machine-specific paths have been replaced with repository-relative configuration, and saved outputs are intentionally cleared.

# Future-Compatible Retrieval
## Same-Channel and Fixed-Scale Mechanism Study

This experiment does **not** change the reranker architecture. It diagnoses two possible causes of the initial Weather behavior:

\[
\boxed{\text{candidate semantic compatibility}\times\text{future-target scaling}}.
\]

# 1. Two experimental axes

## Axis A — Candidate pool

**Cross-channel** allows historical windows from all channels. **Same-channel** restricts candidates to the same physical variable/channel as the query (e.g., temperature queries retrieve historical temperature windows only). The latter is a more semantically controlled definition for datasets such as Weather whose channels represent different physical variables.

## Axis B — Future target

The original local-scale target is

\[
\mathbf y^{\mathrm{local}}=
\frac{\mathbf x_{t+1:t+H}^{\mathrm{norm}}-x_t^{\mathrm{norm}}}
{\sigma(\mathbf x_{t-L+1:t}^{\mathrm{norm}})}.
\]

It provides local scale invariance but can become numerically extreme when the past window is nearly constant. The fixed train-scale target is

\[
\boxed{\mathbf y^{\mathrm{fixed}}=
\mathbf x_{t+1:t+H}^{\mathrm{norm}}-x_t^{\mathrm{norm}}},
\]

where each channel has already been normalized using training statistics. This uses no test information and avoids a local small denominator.

# 2. Full 2 x 2 design

For Weather and ETTh1 at \(H\in\{24,48,96\}\), compare:

1. `Cross_Local`
2. `Cross_Fixed`
3. `Same_Local`
4. `Same_Fixed`

The learned retriever uses three seeds in this mechanism study.

# 3. Strong negative control

For the main `Same_Fixed` configuration, train a Shuffled-Future control in which query futures are cyclically permuted **within the same channel**. This preserves variable-specific future marginals, the candidate pool, observable inputs, architecture, and optimization while breaking only the correct query--future correspondence.

If the correct Future-Compatible model beats this control, the gain cannot be explained solely by same-channel restriction or model capacity.

# 4. Main hypotheses

- **Numerical robustness:** fixed train-scale targets should remove the extreme Weather target magnitudes caused by local small denominators.
- **Semantic compatibility:** same-channel retrieval may be more appropriate than cross-channel retrieval when channels represent heterogeneous physical variables.
- **Future supervision:** `Same_Fixed Learned < Same_Fixed Shuffled` supports the value of correct privileged-future supervision.
- **Generality:** ETTh1 should retain Pattern-to-Learned gains even without cross-channel candidate mixing.

# 5. Interpretation rule

`Local` and `Fixed` use different target definitions, so their raw MSE magnitudes should not be compared directly. Use tail diagnostics, relative Pattern-to-Learned improvements, ranking metrics, and within-target comparisons instead.

> **Public repository version.** Paths are repository-relative by default.
> Set `WHM_DATA_ROOT` to use datasets stored elsewhere and `WHM_WORK_ROOT` to move generated caches/checkpoints outside the repository.
> Saved execution outputs were cleared intentionally so the notebook does not expose machine-specific paths or stale results.


In [ ]:
from pathlib import Path
import os

def _find_repo_root(start=None):
    """Locate the repository root from the current working directory."""
    start = Path(start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "README.md").exists() and (candidate / "experiments").exists():
            return candidate
    raise RuntimeError(
        "Repository root not found. Start Jupyter from inside the cloned "
        "which-histories-matter repository, or set WHM_REPO_ROOT."
    )

_env_repo = os.environ.get("WHM_REPO_ROOT")
REPO_ROOT = Path(_env_repo).expanduser().resolve() if _env_repo else _find_repo_root()
REPO_DATA_ROOT = Path(os.environ.get("WHM_DATA_ROOT", REPO_ROOT / "data")).expanduser().resolve()
REPO_WORK_ROOT = Path(os.environ.get("WHM_WORK_ROOT", REPO_ROOT / "_work")).expanduser().resolve()
REPO_WORK_ROOT.mkdir(parents=True, exist_ok=True)

print("Repository root:", REPO_ROOT)
print("Data root:", REPO_DATA_ROOT)
print("Work root:", REPO_WORK_ROOT)


## 0. Configuration

In [ ]:

from pathlib import Path
import copy
import gc
import math
import random
import warnings

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F

warnings.filterwarnings("ignore")

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)

DATA_PATHS = {
    "ETTh1":
        REPO_DATA_ROOT / "ETT-small/ETTh1.csv",

    "Weather":
        REPO_DATA_ROOT / "weather/weather.csv",
}

PREV_DIR = REPO_WORK_ROOT / "cross_domain_clean"

PREV_CACHE = (
    PREV_DIR /
    "cache"
)

PREV_MODEL_DIR = (
    PREV_DIR /
    "models"
)

RESULT_DIR = REPO_WORK_ROOT / "semantic_compatibility"

CACHE_DIR = (
    RESULT_DIR /
    "cache"
)

MODEL_DIR = (
    RESULT_DIR /
    "models"
)

for p in [
    RESULT_DIR,
    CACHE_DIR,
    MODEL_DIR,
]:
    p.mkdir(
        parents=True,
        exist_ok=True,
    )

DATASET_NAMES = [
    "ETTh1",
    "Weather",
]

HORIZONS = [
    24,
    48,
    96,
]

SEQ_LEN = 96

TOP_M = 100
TOP_K = 10

TAU_Y = 0.50

SEEDS = [
    0,
    1,
    2,
]

TRAIN_BATCH = 128
EVAL_BATCH = 256
SEARCH_QUERY_BATCH = 512

MAX_EPOCHS = 30
PATIENCE = 6

LR = 1e-3
WEIGHT_DECAY = 1e-4

MAX_MEMORY_WINDOWS = 50000
MAX_TRAIN_QUERIES = 6000
MAX_VAL_QUERIES = 6000
MAX_TEST_QUERIES = 8000

BLOCK_ANCHORS = 10
N_BOOT = 5000

EPS = 1e-8

DATASET_SEED = {
    "ETTh1": 1101,
    "Weather": 2202,
}

CONFIGS = [
    "Cross_Local",
    "Cross_Fixed",
    "Same_Local",
    "Same_Fixed",
]

FORCE_REBUILD_SAME_POOL = False
FORCE_RETRAIN = False

print(
    "Device:",
    DEVICE
)

if torch.cuda.is_available():

    print(
        "GPU:",
        torch.cuda.get_device_name(
            0
        )
    )

    print(
        "GPU memory GB:",
        round(
            torch.cuda.get_device_properties(
                0
            ).total_memory /
            1024**3,
            1,
        )
    )

print(
    "Previous experiment:",
    PREV_DIR
)

print(
    "Output:",
    RESULT_DIR
)


## 1. Verify prerequisites

In [ ]:

assert PREV_DIR.exists(), (
    "Run future_compatible_retrieval_cross_domain_clean first."
)

for dataset_name in DATASET_NAMES:

    assert DATA_PATHS[
        dataset_name
    ].exists()

    for H in HORIZONS:

        assert (
            PREV_CACHE /
            f"{dataset_name}_H{H}_windows.npz"
        ).exists()

        assert (
            PREV_CACHE /
            f"{dataset_name}_H{H}_meta.parquet"
        ).exists()

        assert (
            PREV_CACHE /
            f"{dataset_name}_H{H}_topM.npz"
        ).exists()

        for seed in SEEDS:

            assert (
                PREV_MODEL_DIR /
                f"{dataset_name}_H{H}_Learned_seed{seed}.pt"
            ).exists()

print(
    "All previous caches and Cross_Local checkpoints found."
)


## 2. Load raw numeric data

In [ ]:

def load_numeric_csv(
    path,
):
    df = pd.read_csv(
        path
    )

    timestamp_cols = [
        c
        for c in df.columns
        if str(
            c
        ).lower() in {
            "date",
            "datetime",
            "timestamp",
            "time",
        }
    ]

    x = (
        df
        .drop(
            columns=timestamp_cols,
            errors="ignore",
        )
        .apply(
            pd.to_numeric,
            errors="coerce",
        )
    )

    good_cols = [
        c
        for c in x.columns
        if x[
            c
        ].notna().mean() >
        0.99
    ]

    x = (
        x[
            good_cols
        ]
        .replace(
            [
                np.inf,
                -np.inf,
            ],
            np.nan,
        )
        .interpolate(
            axis=0,
            limit_direction="both",
        )
        .ffill()
        .bfill()
    )

    assert np.isfinite(
        x.to_numpy(
            dtype=np.float32
        )
    ).all()

    return x


RAW = {
    name:
        load_numeric_csv(
            DATA_PATHS[
                name
            ]
        )
    for name in DATASET_NAMES
}

for name, df in RAW.items():

    print(
        name,
        df.shape,
        "| channels:",
        list(
            df.columns
        )
    )


## 3. Reconstruct train normalization and splits

In [ ]:

def split_boundaries(
    dataset_name,
    n,
):
    if dataset_name == "ETTh1":

        train_end = (
            12 *
            30 *
            24
        )

        val_end = (
            train_end +
            4 *
            30 *
            24
        )

    else:

        train_end = int(
            0.70 *
            n
        )

        val_end = int(
            0.80 *
            n
        )

    return {
        "train_end":
            train_end,

        "val_end":
            val_end,

        "test_end":
            n,

        "inner_memory_end":
            int(
                0.60 *
                train_end
            ),
    }


SPLITS = {}
CHANNEL_NORMALIZED = {}
CHANNEL_TRAIN_STD_RAW = {}

for dataset_name in DATASET_NAMES:

    df = RAW[
        dataset_name
    ]

    s = split_boundaries(
        dataset_name,
        len(
            df
        ),
    )

    SPLITS[
        dataset_name
    ] = s

    train = df.iloc[
        :s[
            "train_end"
        ]
    ]

    mean = train.mean(
        axis=0
    ).to_numpy(
        dtype=np.float32
    )

    std = train.std(
        axis=0,
        ddof=0,
    ).to_numpy(
        dtype=np.float32
    )

    std = np.where(
        std <
        1e-6,
        1.0,
        std,
    ).astype(
        np.float32
    )

    arr = df.to_numpy(
        dtype=np.float32
    )

    CHANNEL_NORMALIZED[
        dataset_name
    ] = (
        (
            arr -
            mean[
                None,
                :
            ]
        ) /
        std[
            None,
            :
        ]
    ).astype(
        np.float32
    )

    CHANNEL_TRAIN_STD_RAW[
        dataset_name
    ] = std

    print(
        dataset_name,
        s
    )


## 4. Load previous windows and Cross-channel Top-M pools

In [ ]:

WINDOWS = {}
CROSS_PRESELECT = {}

for dataset_name in DATASET_NAMES:

    for H in HORIZONS:

        key = (
            dataset_name,
            H
        )

        z = np.load(
            PREV_CACHE /
            f"{dataset_name}_H{H}_windows.npz"
        )

        WINDOWS[
            key
        ] = {
            "meta":
                pd.read_parquet(
                    PREV_CACHE /
                    f"{dataset_name}_H{H}_meta.parquet"
                ),

            "pattern":
                z[
                    "pattern"
                ].astype(
                    np.float32
                ),

            "context":
                z[
                    "context"
                ].astype(
                    np.float32
                ),

            "local_future":
                z[
                    "future"
                ].astype(
                    np.float32
                ),
        }

        p = np.load(
            PREV_CACHE /
            f"{dataset_name}_H{H}_topM.npz"
        )

        CROSS_PRESELECT[
            key
        ] = {
            k:
                p[
                    k
                ]
            for k in p.files
        }

print(
    "Previous windows and Cross-channel pools loaded."
)


## 5. Reconstruct exact sampled memory/query indices

In [ ]:

def full_phase_indices(
    dataset_name,
    H,
):
    meta = WINDOWS[
        (
            dataset_name,
            H
        )
    ][
        "meta"
    ]

    s = SPLITS[
        dataset_name
    ]

    anchor = meta[
        "Anchor"
    ].to_numpy()

    future_end = meta[
        "FutureEnd"
    ].to_numpy()

    return {
        "train_memory":
            np.where(
                future_end <
                s[
                    "inner_memory_end"
                ]
            )[0],

        "train_query":
            np.where(
                (
                    anchor >=
                    s[
                        "inner_memory_end"
                    ]
                )
                &
                (
                    future_end <
                    s[
                        "train_end"
                    ]
                )
            )[0],

        "val_memory":
            np.where(
                future_end <
                s[
                    "train_end"
                ]
            )[0],

        "val_query":
            np.where(
                (
                    anchor >=
                    s[
                        "train_end"
                    ]
                )
                &
                (
                    future_end <
                    s[
                        "val_end"
                    ]
                )
            )[0],

        "test_memory":
            np.where(
                future_end <
                s[
                    "val_end"
                ]
            )[0],

        "test_query":
            np.where(
                anchor >=
                s[
                    "val_end"
                ]
            )[0],
    }


def deterministic_subset(
    indices,
    max_n,
    seed,
):
    indices = np.asarray(
        indices,
        dtype=np.int64,
    )

    if len(
        indices
    ) <= max_n:

        return np.sort(
            indices
        )

    rng = np.random.default_rng(
        seed
    )

    return np.sort(
        rng.choice(
            indices,
            size=max_n,
            replace=False,
        )
    )


TASK_INDICES = {}

for dataset_name in DATASET_NAMES:

    for H in HORIZONS:

        key = (
            dataset_name,
            H
        )

        p = full_phase_indices(
            dataset_name,
            H,
        )

        base = (
            DATASET_SEED[
                dataset_name
            ] +
            H *
            10
        )

        sampled = {
            "train_memory":
                deterministic_subset(
                    p[
                        "train_memory"
                    ],
                    MAX_MEMORY_WINDOWS,
                    base +
                    1,
                ),

            "train_query":
                deterministic_subset(
                    p[
                        "train_query"
                    ],
                    MAX_TRAIN_QUERIES,
                    base +
                    2,
                ),

            "val_memory":
                deterministic_subset(
                    p[
                        "val_memory"
                    ],
                    MAX_MEMORY_WINDOWS,
                    base +
                    3,
                ),

            "val_query":
                deterministic_subset(
                    p[
                        "val_query"
                    ],
                    MAX_VAL_QUERIES,
                    base +
                    4,
                ),

            "test_memory":
                deterministic_subset(
                    p[
                        "test_memory"
                    ],
                    MAX_MEMORY_WINDOWS,
                    base +
                    5,
                ),

            "test_query":
                deterministic_subset(
                    p[
                        "test_query"
                    ],
                    MAX_TEST_QUERIES,
                    base +
                    6,
                ),
        }

        TASK_INDICES[
            key
        ] = sampled

        cross = CROSS_PRESELECT[
            key
        ]

        for phase in [
            "train",
            "val",
            "test",
        ]:

            np.testing.assert_array_equal(
                cross[
                    f"{phase}_query"
                ],
                sampled[
                    f"{phase}_query"
                ],
            )

print(
    "Exact sampling reconstruction passed."
)


## 6. Reconstruct Fixed train-scale future targets

In [ ]:

FIXED_FUTURE = {}
PAST_LOCAL_STD = {}

for dataset_name in DATASET_NAMES:

    arr = CHANNEL_NORMALIZED[
        dataset_name
    ]

    for H in HORIZONS:

        key = (
            dataset_name,
            H
        )

        meta = WINDOWS[
            key
        ][
            "meta"
        ]

        fixed = np.empty(
            (
                len(
                    meta
                ),
                H,
            ),
            dtype=np.float32,
        )

        local_std = np.empty(
            len(
                meta
            ),
            dtype=np.float32,
        )

        for j, row in enumerate(
            meta.itertuples(
                index=False
            )
        ):

            cidx = int(
                row.ChannelIndex
            )

            anchor = int(
                row.Anchor
            )

            series = arr[
                :,
                cidx
            ]

            past = series[
                anchor -
                SEQ_LEN:
                anchor
            ]

            future = series[
                anchor:
                anchor +
                H
            ]

            s = float(
                np.std(
                    past
                )
            )

            local_std[
                j
            ] = s

            fixed[
                j
            ] = (
                future -
                past[
                    -1
                ]
            )

        reconstructed_local = (
            fixed /
            (
                local_std[
                    :,
                    None
                ] +
                EPS
            )
        )

        max_diff = float(
            np.max(
                np.abs(
                    reconstructed_local -
                    WINDOWS[
                        key
                    ][
                        "local_future"
                    ]
                )
            )
        )

        assert max_diff < 1e-3, (
            dataset_name,
            H,
            max_diff,
        )

        FIXED_FUTURE[
            key
        ] = fixed

        PAST_LOCAL_STD[
            key
        ] = local_std

        print(
            dataset_name,
            "H=",
            H,
            "| local reconstruction max diff",
            f"{max_diff:.2e}",
        )


## 7. Target magnitude diagnostics

In [ ]:

target_diag_rows = []

for dataset_name in DATASET_NAMES:

    for H in HORIZONS:

        key = (
            dataset_name,
            H
        )

        test_idx = TASK_INDICES[
            key
        ][
            "test_query"
        ]

        local = WINDOWS[
            key
        ][
            "local_future"
        ][
            test_idx
        ]

        fixed = FIXED_FUTURE[
            key
        ][
            test_idx
        ]

        local_mag = np.max(
            np.abs(
                local
            ),
            axis=1,
        )

        fixed_mag = np.max(
            np.abs(
                fixed
            ),
            axis=1,
        )

        std = PAST_LOCAL_STD[
            key
        ][
            test_idx
        ]

        target_diag_rows.append({
            "Dataset":
                dataset_name,

            "Horizon":
                H,

            "PastLocalStd_Q01":
                float(
                    np.quantile(
                        std,
                        0.01,
                    )
                ),

            "PastLocalStd_Q05":
                float(
                    np.quantile(
                        std,
                        0.05,
                    )
                ),

            "PastLocalStd_Median":
                float(
                    np.median(
                        std
                    )
                ),

            "LocalTargetMaxAbs_Q95":
                float(
                    np.quantile(
                        local_mag,
                        0.95,
                    )
                ),

            "LocalTargetMaxAbs_Q99":
                float(
                    np.quantile(
                        local_mag,
                        0.99,
                    )
                ),

            "LocalTargetMaxAbs_Max":
                float(
                    np.max(
                        local_mag
                    )
                ),

            "FixedTargetMaxAbs_Q95":
                float(
                    np.quantile(
                        fixed_mag,
                        0.95,
                    )
                ),

            "FixedTargetMaxAbs_Q99":
                float(
                    np.quantile(
                        fixed_mag,
                        0.99,
                    )
                ),

            "FixedTargetMaxAbs_Max":
                float(
                    np.max(
                        fixed_mag
                    )
                ),
        })

target_diag = pd.DataFrame(
    target_diag_rows
)

display(
    target_diag
)

target_diag.to_csv(
    RESULT_DIR /
    "01_target_scaling_diagnostics.csv",
    index=False,
)


## 8. Reconstruct train-memory context scaler

In [ ]:

def fit_robust_scaler(
    x,
):
    med = np.median(
        x,
        axis=0,
    )

    q25 = np.percentile(
        x,
        25,
        axis=0,
    )

    q75 = np.percentile(
        x,
        75,
        axis=0,
    )

    iqr = (
        q75 -
        q25
    )

    iqr = np.where(
        iqr <
        1e-5,
        1.0,
        iqr,
    )

    return (
        med.astype(
            np.float32
        ),
        iqr.astype(
            np.float32
        ),
    )


def apply_robust_scaler(
    x,
    med,
    iqr,
):
    z = (
        x -
        med
    ) / iqr

    z = np.clip(
        z,
        -8.0,
        8.0,
    )

    return z.astype(
        np.float32
    )


CONTEXT_SCALED = {}

for dataset_name in DATASET_NAMES:

    for H in HORIZONS:

        key = (
            dataset_name,
            H
        )

        w = WINDOWS[
            key
        ]

        train_mem = TASK_INDICES[
            key
        ][
            "train_memory"
        ]

        med, iqr = fit_robust_scaler(
            w[
                "context"
            ][
                train_mem
            ]
        )

        CONTEXT_SCALED[
            key
        ] = apply_robust_scaler(
            w[
                "context"
            ],
            med,
            iqr,
        )


## 9. GPU similarity search

In [ ]:

@torch.no_grad()
def cosine_topm(
    candidate_pattern,
    query_pattern,
    top_m,
):
    cand = torch.tensor(
        candidate_pattern,
        dtype=torch.float32,
        device=DEVICE,
    )

    query = torch.tensor(
        query_pattern,
        dtype=torch.float32,
        device=DEVICE,
    )

    idx_list = []
    score_list = []

    for start in range(
        0,
        len(
            query_pattern
        ),
        SEARCH_QUERY_BATCH,
    ):

        end = min(
            start +
            SEARCH_QUERY_BATCH,
            len(
                query_pattern
            ),
        )

        sim = (
            query[
                start:end
            ]
            @
            cand.T
        )

        score, idx = torch.topk(
            sim,
            k=top_m,
            dim=1,
            largest=True,
        )

        idx_list.append(
            idx.cpu()
        )

        score_list.append(
            score.cpu()
        )

        del sim

    return (
        torch.cat(
            idx_list,
            dim=0,
        ).numpy().astype(
            np.int64
        ),

        torch.cat(
            score_list,
            dim=0,
        ).numpy().astype(
            np.float32
        ),
    )


## 10. Build Same-channel Pattern Top-M pools

In [ ]:

SAME_PRESELECT = {}
pool_rows = []

for dataset_name in DATASET_NAMES:

    for H in HORIZONS:

        key = (
            dataset_name,
            H
        )

        cache_path = (
            CACHE_DIR /
            f"{dataset_name}_H{H}_same_channel_topM.npz"
        )

        if (
            cache_path.exists()
            and
            not
            FORCE_REBUILD_SAME_POOL
        ):

            z = np.load(
                cache_path
            )

            out = {
                k:
                    z[
                        k
                    ]
                for k in z.files
            }

            print(
                "Loaded Same-channel pool:",
                dataset_name,
                H
            )

        else:

            w = WINDOWS[
                key
            ]

            meta = w[
                "meta"
            ]

            sampled = TASK_INDICES[
                key
            ]

            out = {}

            for phase in [
                "train",
                "val",
                "test",
            ]:

                mem_idx = sampled[
                    f"{phase}_memory"
                ]

                q_idx = sampled[
                    f"{phase}_query"
                ]

                mem_channel = meta.iloc[
                    mem_idx
                ][
                    "ChannelIndex"
                ].to_numpy(
                    dtype=np.int64
                )

                q_channel = meta.iloc[
                    q_idx
                ][
                    "ChannelIndex"
                ].to_numpy(
                    dtype=np.int64
                )

                result_idx = np.empty(
                    (
                        len(
                            q_idx
                        ),
                        TOP_M,
                    ),
                    dtype=np.int64,
                )

                result_score = np.empty(
                    (
                        len(
                            q_idx
                        ),
                        TOP_M,
                    ),
                    dtype=np.float32,
                )

                unique_channels = np.unique(
                    q_channel
                )

                min_candidates = math.inf
                max_candidates = 0

                for c in unique_channels:

                    q_pos = np.where(
                        q_channel ==
                        c
                    )[0]

                    mem_pos = np.where(
                        mem_channel ==
                        c
                    )[0]

                    assert len(
                        mem_pos
                    ) >= TOP_M, (
                        dataset_name,
                        H,
                        phase,
                        c,
                        len(
                            mem_pos
                        ),
                    )

                    min_candidates = min(
                        min_candidates,
                        len(
                            mem_pos
                        )
                    )

                    max_candidates = max(
                        max_candidates,
                        len(
                            mem_pos
                        )
                    )

                    local_idx, score = cosine_topm(
                        w[
                            "pattern"
                        ][
                            mem_idx[
                                mem_pos
                            ]
                        ],
                        w[
                            "pattern"
                        ][
                            q_idx[
                                q_pos
                            ]
                        ],
                        TOP_M,
                    )

                    result_idx[
                        q_pos
                    ] = mem_idx[
                        mem_pos[
                            local_idx
                        ]
                    ]

                    result_score[
                        q_pos
                    ] = score

                out[
                    f"{phase}_idx"
                ] = result_idx

                out[
                    f"{phase}_score"
                ] = result_score

                out[
                    f"{phase}_query"
                ] = q_idx

                pool_rows.append({
                    "Dataset":
                        dataset_name,

                    "Horizon":
                        H,

                    "Phase":
                        phase,

                    "Queries":
                        len(
                            q_idx
                        ),

                    "MinSameChannelMemoryCandidates":
                        int(
                            min_candidates
                        ),

                    "MaxSameChannelMemoryCandidates":
                        int(
                            max_candidates
                        ),
                })

            np.savez_compressed(
                cache_path,
                **out,
            )

            print(
                "Built Same-channel pool:",
                dataset_name,
                H
            )

        SAME_PRESELECT[
            key
        ] = out

same_pool_summary = pd.DataFrame(
    pool_rows
)

if len(
    same_pool_summary
) > 0:

    display(
        same_pool_summary
    )

    same_pool_summary.to_csv(
        RESULT_DIR /
        "02_same_channel_pool_summary.csv",
        index=False,
    )


## 11. Phase packaging by candidate mode and target mode

In [ ]:

def get_preselect(
    dataset_name,
    H,
    candidate_mode,
):
    key = (
        dataset_name,
        H
    )

    if candidate_mode == "Cross":

        return CROSS_PRESELECT[
            key
        ]

    if candidate_mode == "Same":

        return SAME_PRESELECT[
            key
        ]

    raise ValueError(
        candidate_mode
    )


def get_future(
    dataset_name,
    H,
    target_mode,
):
    key = (
        dataset_name,
        H
    )

    if target_mode == "Local":

        return WINDOWS[
            key
        ][
            "local_future"
        ]

    if target_mode == "Fixed":

        return FIXED_FUTURE[
            key
        ]

    raise ValueError(
        target_mode
    )


def phase_data(
    dataset_name,
    H,
    phase,
    candidate_mode,
    target_mode,
):
    key = (
        dataset_name,
        H
    )

    w = WINDOWS[
        key
    ]

    p = get_preselect(
        dataset_name,
        H,
        candidate_mode,
    )

    future = get_future(
        dataset_name,
        H,
        target_mode,
    )

    q_idx = p[
        f"{phase}_query"
    ]

    cand_idx = p[
        f"{phase}_idx"
    ]

    meta = w[
        "meta"
    ]

    return {
        "q_idx":
            q_idx,

        "cand_idx":
            cand_idx,

        "pattern_score":
            p[
                f"{phase}_score"
            ].astype(
                np.float32
            ),

        "q_context":
            CONTEXT_SCALED[
                key
            ][
                q_idx
            ],

        "cand_context":
            CONTEXT_SCALED[
                key
            ][
                cand_idx
            ],

        "q_future":
            future[
                q_idx
            ],

        "cand_future":
            future[
                cand_idx
            ],

        "q_anchor":
            meta.iloc[
                q_idx
            ][
                "Anchor"
            ].to_numpy(
                dtype=np.int64
            ),

        "q_channel":
            meta.iloc[
                q_idx
            ][
                "ChannelIndex"
            ].to_numpy(
                dtype=np.int64
            ),

        "cand_channel":
            meta.iloc[
                cand_idx.reshape(
                    -1
                )
            ][
                "ChannelIndex"
            ].to_numpy(
                dtype=np.int64
            ).reshape(
                cand_idx.shape
            ),
    }


## 12. Retrieval metrics

In [ ]:

def future_distance(
    q_future,
    cand_future,
):
    return np.mean(
        (
            cand_future -
            q_future[
                :,
                None,
                :
            ]
        ) ** 2,
        axis=2,
    )


def topk_from_scores(
    score,
    k,
):
    idx = np.argpartition(
        -score,
        kth=k -
        1,
        axis=1,
    )[
        :,
        :k
    ]

    row = np.arange(
        len(
            score
        )
    )[
        :,
        None
    ]

    local_score = score[
        row,
        idx
    ]

    order = np.argsort(
        -local_score,
        axis=1,
    )

    return idx[
        row,
        order
    ]


def gather2(
    x,
    idx,
):
    row = np.arange(
        len(
            x
        )
    )[
        :,
        None
    ]

    return x[
        row,
        idx
    ]


def gather3(
    x,
    idx,
):
    row = np.arange(
        len(
            x
        )
    )[
        :,
        None
    ]

    return x[
        row,
        idx,
        :
    ]


def ndcg_at_k(
    score,
    fdist,
    k,
):
    mean = fdist.mean(
        axis=1,
        keepdims=True,
    )

    std = (
        fdist.std(
            axis=1,
            keepdims=True,
        ) +
        1e-6
    )

    z = (
        fdist -
        mean
    ) / std

    relevance = np.exp(
        -z /
        TAU_Y
    )

    selected = topk_from_scores(
        score,
        k,
    )

    ideal = np.argsort(
        -relevance,
        axis=1,
    )[
        :,
        :k
    ]

    rel_sel = gather2(
        relevance,
        selected,
    )

    rel_ideal = gather2(
        relevance,
        ideal,
    )

    discount = (
        1.0 /
        np.log2(
            np.arange(
                2,
                k +
                2
            )
        )
    )[
        None,
        :
    ]

    dcg = np.sum(
        rel_sel *
        discount,
        axis=1,
    )

    idcg = (
        np.sum(
            rel_ideal *
            discount,
            axis=1,
        ) +
        EPS
    )

    return (
        dcg /
        idcg
    ).astype(
        np.float32
    )


def oracle_recall_at_k(
    selected,
    fdist,
    k,
):
    oracle = np.argpartition(
        fdist,
        kth=k -
        1,
        axis=1,
    )[
        :,
        :k
    ]

    out = np.empty(
        len(
            selected
        ),
        dtype=np.float32,
    )

    for i in range(
        len(
            selected
        )
    ):

        out[
            i
        ] = (
            len(
                set(
                    selected[
                        i
                    ].tolist()
                )
                &
                set(
                    oracle[
                        i
                    ].tolist()
                )
            )
            /
            k
        )

    return out


def query_metrics(
    score,
    phase,
):
    fdist = future_distance(
        phase[
            "q_future"
        ],
        phase[
            "cand_future"
        ],
    )

    selected = topk_from_scores(
        score,
        TOP_K,
    )

    selected_dist = gather2(
        fdist,
        selected,
    )

    selected_future = gather3(
        phase[
            "cand_future"
        ],
        selected,
    )

    pred = selected_future.mean(
        axis=1
    )

    forecast_mse = np.mean(
        (
            pred -
            phase[
                "q_future"
            ]
        ) ** 2,
        axis=1,
    )

    selected_channel = gather2(
        phase[
            "cand_channel"
        ],
        selected,
    )

    other_frac = np.mean(
        (
            selected_channel
            !=
            phase[
                "q_channel"
            ][
                :,
                None
            ]
        ),
        axis=1,
    )

    return pd.DataFrame({
        "AnalogFutureMSE":
            selected_dist.mean(
                axis=1
            ).astype(
                np.float32
            ),

        "RetrievalForecastMSE":
            forecast_mse.astype(
                np.float32
            ),

        "NDCG@K":
            ndcg_at_k(
                score,
                fdist,
                TOP_K,
            ),

        "OracleRecall@K":
            oracle_recall_at_k(
                selected,
                fdist,
                TOP_K,
            ),

        "OtherChannelFrac@K":
            other_frac.astype(
                np.float32
            ),
    })


def oracle_within_m_metrics(
    phase,
):
    fdist = future_distance(
        phase[
            "q_future"
        ],
        phase[
            "cand_future"
        ],
    )

    idx = np.argpartition(
        fdist,
        kth=TOP_K -
        1,
        axis=1,
    )[
        :,
        :TOP_K
    ]

    dist = gather2(
        fdist,
        idx,
    ).mean(
        axis=1
    )

    future = gather3(
        phase[
            "cand_future"
        ],
        idx,
    )

    pred = future.mean(
        axis=1
    )

    forecast_mse = np.mean(
        (
            pred -
            phase[
                "q_future"
            ]
        ) ** 2,
        axis=1,
    )

    return pd.DataFrame({
        "OracleAnalogFutureMSE":
            dist.astype(
                np.float32
            ),

        "OracleRetrievalForecastMSE":
            forecast_mse.astype(
                np.float32
            ),
    })


## 13. Future-Compatible reranker

In [ ]:

class FutureCompatibleReranker(
    nn.Module
):
    def __init__(
        self,
        context_dim=7,
        hidden_dim=128,
        dropout=0.10,
        initial_alpha=0.10,
    ):
        super().__init__()

        input_dim = (
            1 +
            4 *
            context_dim
        )

        self.mlp = nn.Sequential(
            nn.Linear(
                input_dim,
                hidden_dim,
            ),
            nn.LayerNorm(
                hidden_dim
            ),
            nn.GELU(),
            nn.Dropout(
                dropout
            ),
            nn.Linear(
                hidden_dim,
                hidden_dim //
                2,
            ),
            nn.GELU(),
            nn.Dropout(
                dropout
            ),
            nn.Linear(
                hidden_dim //
                2,
                1,
            ),
        )

        raw_alpha = math.log(
            math.exp(
                initial_alpha
            ) -
            1.0
        )

        self.raw_alpha = nn.Parameter(
            torch.tensor(
                raw_alpha,
                dtype=torch.float32,
            )
        )

    @property
    def alpha(
        self
    ):
        return F.softplus(
            self.raw_alpha
        )

    def forward(
        self,
        pattern_score,
        q_context,
        cand_context,
    ):
        B, M, D = (
            cand_context.shape
        )

        q = (
            q_context[
                :,
                None,
                :
            ]
            .expand(
                -1,
                M,
                -1,
            )
        )

        diff = (
            q -
            cand_context
        )

        feat = torch.cat(
            [
                pattern_score[
                    ...,
                    None
                ],
                q,
                cand_context,
                diff,
                diff.abs(),
            ],
            dim=-1,
        )

        delta = (
            self.mlp(
                feat
            )
            .squeeze(
                -1
            )
        )

        return (
            pattern_score +
            self.alpha *
            delta
        )


## 14. Training helpers

In [ ]:

def set_seed(
    seed,
):
    random.seed(
        seed
    )

    np.random.seed(
        seed
    )

    torch.manual_seed(
        seed
    )

    if torch.cuda.is_available():

        torch.cuda.manual_seed_all(
            seed
        )


def listwise_future_loss(
    score,
    future_dist,
):
    mean = future_dist.mean(
        dim=1,
        keepdim=True,
    )

    std = future_dist.std(
        dim=1,
        keepdim=True,
        unbiased=False,
    ).clamp_min(
        1e-6
    )

    z = (
        future_dist -
        mean
    ) / std

    target = torch.softmax(
        -z /
        TAU_Y,
        dim=1,
    )

    log_prob = F.log_softmax(
        score,
        dim=1,
    )

    loss = -(
        target *
        log_prob
    ).sum(
        dim=1
    ).mean()

    assert torch.isfinite(
        loss
    )

    return loss


def channelwise_shuffled_future(
    phase,
    seed,
):
    """
    Strong negative control:
    shuffle query futures only within the same physical channel.
    """
    rng = np.random.default_rng(
        seed
    )

    out = phase[
        "q_future"
    ].copy()

    channels = phase[
        "q_channel"
    ]

    for c in np.unique(
        channels
    ):

        pos = np.where(
            channels ==
            c
        )[0]

        if len(
            pos
        ) <= 1:
            continue

        perm = rng.permutation(
            pos
        )

        out[
            pos
        ] = phase[
            "q_future"
        ][
            perm
        ]

    return out


def train_epoch(
    model,
    optimizer,
    phase,
    shuffled_future=None,
):
    model.train()

    n = len(
        phase[
            "q_idx"
        ]
    )

    order = np.random.permutation(
        n
    )

    losses = []

    for start in range(
        0,
        n,
        TRAIN_BATCH,
    ):

        ids = order[
            start:
            start +
            TRAIN_BATCH
        ]

        ps = torch.tensor(
            phase[
                "pattern_score"
            ][
                ids
            ],
            dtype=torch.float32,
            device=DEVICE,
        )

        qc = torch.tensor(
            phase[
                "q_context"
            ][
                ids
            ],
            dtype=torch.float32,
            device=DEVICE,
        )

        cc = torch.tensor(
            phase[
                "cand_context"
            ][
                ids
            ],
            dtype=torch.float32,
            device=DEVICE,
        )

        cf = torch.tensor(
            phase[
                "cand_future"
            ][
                ids
            ],
            dtype=torch.float32,
            device=DEVICE,
        )

        qf_np = (
            phase[
                "q_future"
            ][
                ids
            ]
            if shuffled_future is None
            else
            shuffled_future[
                ids
            ]
        )

        qf = torch.tensor(
            qf_np,
            dtype=torch.float32,
            device=DEVICE,
        )

        fdist = (
            (
                cf -
                qf[
                    :,
                    None,
                    :
                ]
            ) ** 2
        ).mean(
            dim=2
        )

        optimizer.zero_grad(
            set_to_none=True
        )

        score = model(
            ps,
            qc,
            cc,
        )

        loss = listwise_future_loss(
            score,
            fdist,
        )

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            5.0,
        )

        optimizer.step()

        losses.append(
            float(
                loss.item()
            )
        )

    return float(
        np.mean(
            losses
        )
    )


@torch.no_grad()
def predict_scores(
    model,
    phase,
):
    model.eval()

    chunks = []

    n = len(
        phase[
            "q_idx"
        ]
    )

    for start in range(
        0,
        n,
        EVAL_BATCH,
    ):

        end = min(
            start +
            EVAL_BATCH,
            n,
        )

        ps = torch.tensor(
            phase[
                "pattern_score"
            ][
                start:end
            ],
            dtype=torch.float32,
            device=DEVICE,
        )

        qc = torch.tensor(
            phase[
                "q_context"
            ][
                start:end
            ],
            dtype=torch.float32,
            device=DEVICE,
        )

        cc = torch.tensor(
            phase[
                "cand_context"
            ][
                start:end
            ],
            dtype=torch.float32,
            device=DEVICE,
        )

        chunks.append(
            model(
                ps,
                qc,
                cc,
            ).cpu()
        )

    return (
        torch.cat(
            chunks,
            dim=0,
        ).numpy().astype(
            np.float32
        )
    )


## 15. Phase A selection + Phase B refit

In [ ]:

def select_epoch(
    dataset_name,
    H,
    seed,
    candidate_mode,
    target_mode,
    shuffled=False,
):
    set_seed(
        seed
    )

    train_phase = phase_data(
        dataset_name,
        H,
        "train",
        candidate_mode,
        target_mode,
    )

    val_phase = phase_data(
        dataset_name,
        H,
        "val",
        candidate_mode,
        target_mode,
    )

    model = FutureCompatibleReranker().to(
        DEVICE
    )

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=LR,
        weight_decay=WEIGHT_DECAY,
    )

    shuffled_train = None

    if shuffled:

        shuffled_train = channelwise_shuffled_future(
            train_phase,
            seed=(
                100000 +
                seed
            ),
        )

    best_epoch = None
    best_val = float(
        "inf"
    )

    wait = 0
    history = []

    for epoch in range(
        1,
        MAX_EPOCHS +
        1,
    ):

        loss = train_epoch(
            model,
            optimizer,
            train_phase,
            shuffled_future=shuffled_train,
        )

        val_score = predict_scores(
            model,
            val_phase,
        )

        val_m = query_metrics(
            val_score,
            val_phase,
        )

        val_analog = float(
            val_m[
                "AnalogFutureMSE"
            ].mean()
        )

        history.append({
            "Epoch":
                epoch,

            "TrainLoss":
                loss,

            "ValAnalogFutureMSE":
                val_analog,

            "ValNDCG@K":
                float(
                    val_m[
                        "NDCG@K"
                    ].mean()
                ),

            "Alpha":
                float(
                    model.alpha.item()
                ),
        })

        if (
            val_analog <
            best_val -
            1e-10
        ):

            best_val = val_analog
            best_epoch = epoch
            wait = 0

        else:

            wait += 1

        if wait >= PATIENCE:
            break

    assert best_epoch is not None

    return (
        best_epoch,
        best_val,
        pd.DataFrame(
            history
        ),
    )


def refit_model(
    dataset_name,
    H,
    seed,
    epochs,
    candidate_mode,
    target_mode,
    shuffled=False,
):
    set_seed(
        seed
    )

    train_phase = phase_data(
        dataset_name,
        H,
        "train",
        candidate_mode,
        target_mode,
    )

    val_phase = phase_data(
        dataset_name,
        H,
        "val",
        candidate_mode,
        target_mode,
    )

    model = FutureCompatibleReranker().to(
        DEVICE
    )

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=LR,
        weight_decay=WEIGHT_DECAY,
    )

    shuffled_train = None
    shuffled_val = None

    if shuffled:

        shuffled_train = channelwise_shuffled_future(
            train_phase,
            seed=(
                200000 +
                seed
            ),
        )

        shuffled_val = channelwise_shuffled_future(
            val_phase,
            seed=(
                300000 +
                seed
            ),
        )

    for _ in range(
        epochs
    ):

        train_epoch(
            model,
            optimizer,
            train_phase,
            shuffled_future=shuffled_train,
        )

        train_epoch(
            model,
            optimizer,
            val_phase,
            shuffled_future=shuffled_val,
        )

    return model


## 16. Train/load all mechanism models

In [ ]:

MODELS = {}
training_rows = []

for dataset_name in DATASET_NAMES:

    for H in HORIZONS:

        for config in CONFIGS:

            candidate_mode, target_mode = config.split(
                "_"
            )

            for seed in SEEDS:

                # Cross_Local is exactly the previous clean experiment.
                if config == "Cross_Local":

                    path = (
                        PREV_MODEL_DIR /
                        f"{dataset_name}_H{H}_Learned_seed{seed}.pt"
                    )

                    ckpt = torch.load(
                        path,
                        map_location="cpu",
                        weights_only=False,
                    )

                    model = FutureCompatibleReranker().to(
                        DEVICE
                    )

                    model.load_state_dict(
                        ckpt[
                            "StateDict"
                        ]
                    )

                    best_epoch = int(
                        ckpt[
                            "BestEpoch"
                        ]
                    )

                    best_val = float(
                        ckpt[
                            "BestValidationAnalogFutureMSE"
                        ]
                    )

                    source = "previous"

                else:

                    path = (
                        MODEL_DIR /
                        f"{dataset_name}_H{H}_{config}_Learned_seed{seed}.pt"
                    )

                    if (
                        path.exists()
                        and
                        not
                        FORCE_RETRAIN
                    ):

                        ckpt = torch.load(
                            path,
                            map_location="cpu",
                            weights_only=False,
                        )

                        model = FutureCompatibleReranker().to(
                            DEVICE
                        )

                        model.load_state_dict(
                            ckpt[
                                "StateDict"
                            ]
                        )

                        best_epoch = int(
                            ckpt[
                                "BestEpoch"
                            ]
                        )

                        best_val = float(
                            ckpt[
                                "BestValidationAnalogFutureMSE"
                            ]
                        )

                        source = "loaded"

                    else:

                        (
                            best_epoch,
                            best_val,
                            history,
                        ) = select_epoch(
                            dataset_name,
                            H,
                            seed,
                            candidate_mode,
                            target_mode,
                            shuffled=False,
                        )

                        model = refit_model(
                            dataset_name,
                            H,
                            seed,
                            best_epoch,
                            candidate_mode,
                            target_mode,
                            shuffled=False,
                        )

                        torch.save(
                            {
                                "Dataset":
                                    dataset_name,

                                "Horizon":
                                    H,

                                "Config":
                                    config,

                                "Method":
                                    "Learned",

                                "Seed":
                                    seed,

                                "BestEpoch":
                                    best_epoch,

                                "BestValidationAnalogFutureMSE":
                                    best_val,

                                "StateDict":
                                    model.state_dict(),
                            },
                            path,
                        )

                        source = "trained"

                model.eval()

                MODELS[
                    (
                        dataset_name,
                        H,
                        config,
                        "Learned",
                        seed
                    )
                ] = model

                training_rows.append({
                    "Dataset":
                        dataset_name,

                    "Horizon":
                        H,

                    "Config":
                        config,

                    "Method":
                        "Learned",

                    "Seed":
                        seed,

                    "BestEpoch":
                        best_epoch,

                    "BestValidationAnalogFutureMSE":
                        best_val,

                    "Alpha":
                        float(
                            model.alpha.item()
                        ),

                    "Source":
                        source,
                })

                print(
                    dataset_name,
                    "H=",
                    H,
                    config,
                    "Learned",
                    "seed=",
                    seed,
                    "|",
                    source,
                    "| epoch",
                    best_epoch,
                    "| val",
                    best_val,
                )


## 17. Train/load within-channel Shuffled-Future control for Same_Fixed

In [ ]:

for dataset_name in DATASET_NAMES:

    for H in HORIZONS:

        config = "Same_Fixed"

        candidate_mode = "Same"
        target_mode = "Fixed"

        for seed in SEEDS:

            path = (
                MODEL_DIR /
                f"{dataset_name}_H{H}_{config}_Shuffled_seed{seed}.pt"
            )

            if (
                path.exists()
                and
                not
                FORCE_RETRAIN
            ):

                ckpt = torch.load(
                    path,
                    map_location="cpu",
                    weights_only=False,
                )

                model = FutureCompatibleReranker().to(
                    DEVICE
                )

                model.load_state_dict(
                    ckpt[
                        "StateDict"
                    ]
                )

                best_epoch = int(
                    ckpt[
                        "BestEpoch"
                    ]
                )

                best_val = float(
                    ckpt[
                        "BestValidationAnalogFutureMSE"
                    ]
                )

                source = "loaded"

            else:

                (
                    best_epoch,
                    best_val,
                    history,
                ) = select_epoch(
                    dataset_name,
                    H,
                    seed,
                    candidate_mode,
                    target_mode,
                    shuffled=True,
                )

                model = refit_model(
                    dataset_name,
                    H,
                    seed,
                    best_epoch,
                    candidate_mode,
                    target_mode,
                    shuffled=True,
                )

                torch.save(
                    {
                        "Dataset":
                            dataset_name,

                        "Horizon":
                            H,

                        "Config":
                            config,

                        "Method":
                            "ShuffledFutureWithinChannel",

                        "Seed":
                            seed,

                        "BestEpoch":
                            best_epoch,

                        "BestValidationAnalogFutureMSE":
                            best_val,

                        "StateDict":
                            model.state_dict(),
                    },
                    path,
                )

                source = "trained"

            model.eval()

            MODELS[
                (
                    dataset_name,
                    H,
                    config,
                    "Shuffled",
                    seed
                )
            ] = model

            training_rows.append({
                "Dataset":
                    dataset_name,

                "Horizon":
                    H,

                "Config":
                    config,

                "Method":
                    "ShuffledFutureWithinChannel",

                "Seed":
                    seed,

                "BestEpoch":
                    best_epoch,

                "BestValidationAnalogFutureMSE":
                    best_val,

                "Alpha":
                    float(
                        model.alpha.item()
                    ),

                "Source":
                    source,
            })

            print(
                dataset_name,
                "H=",
                H,
                config,
                "Shuffled",
                "seed=",
                seed,
                "|",
                source,
                "| epoch",
                best_epoch,
                "| val",
                best_val,
            )

training_table = pd.DataFrame(
    training_rows
)

training_table.to_csv(
    RESULT_DIR /
    "03_training_summary.csv",
    index=False,
)


## 18. Evaluate Pattern, Learned, Oracle for all 2×2 conditions

In [ ]:

seed_rows = []
main_rows = []
oracle_rows = []
QUERY_RESULTS = {}

for dataset_name in DATASET_NAMES:

    for H in HORIZONS:

        for config in CONFIGS:

            candidate_mode, target_mode = config.split(
                "_"
            )

            test_phase = phase_data(
                dataset_name,
                H,
                "test",
                candidate_mode,
                target_mode,
            )

            pattern_m = query_metrics(
                test_phase[
                    "pattern_score"
                ],
                test_phase,
            )

            oracle_m = oracle_within_m_metrics(
                test_phase
            )

            seed_frames = []

            for seed in SEEDS:

                model = MODELS[
                    (
                        dataset_name,
                        H,
                        config,
                        "Learned",
                        seed
                    )
                ]

                score = predict_scores(
                    model,
                    test_phase,
                )

                m = query_metrics(
                    score,
                    test_phase,
                )

                seed_frames.append(
                    m
                )

                seed_rows.append({
                    "Dataset":
                        dataset_name,

                    "Horizon":
                        H,

                    "Config":
                        config,

                    "Seed":
                        seed,

                    "PatternAnalogFutureMSE":
                        float(
                            pattern_m[
                                "AnalogFutureMSE"
                            ].mean()
                        ),

                    "LearnedAnalogFutureMSE":
                        float(
                            m[
                                "AnalogFutureMSE"
                            ].mean()
                        ),

                    "AnalogImprovement_%":
                        100.0 *
                        (
                            pattern_m[
                                "AnalogFutureMSE"
                            ].mean()
                            -
                            m[
                                "AnalogFutureMSE"
                            ].mean()
                        )
                        /
                        pattern_m[
                            "AnalogFutureMSE"
                        ].mean(),

                    "PatternRetrievalForecastMSE":
                        float(
                            pattern_m[
                                "RetrievalForecastMSE"
                            ].mean()
                        ),

                    "LearnedRetrievalForecastMSE":
                        float(
                            m[
                                "RetrievalForecastMSE"
                            ].mean()
                        ),

                    "LearnedNDCG@K":
                        float(
                            m[
                                "NDCG@K"
                            ].mean()
                        ),

                    "LearnedOracleRecall@K":
                        float(
                            m[
                                "OracleRecall@K"
                            ].mean()
                        ),
                })

            learned_mean = pd.DataFrame({
                col:
                    np.stack(
                        [
                            f[
                                col
                            ].to_numpy()
                            for f in seed_frames
                        ],
                        axis=0,
                    ).mean(
                        axis=0
                    )

                for col in seed_frames[
                    0
                ].columns
            })

            learned_std = pd.DataFrame({
                col:
                    np.stack(
                        [
                            f[
                                col
                            ].to_numpy()
                            for f in seed_frames
                        ],
                        axis=0,
                    ).std(
                        axis=0,
                        ddof=1,
                    )

                for col in seed_frames[
                    0
                ].columns
            })

            q = pd.DataFrame({
                "Anchor":
                    test_phase[
                        "q_anchor"
                    ],

                "ChannelIndex":
                    test_phase[
                        "q_channel"
                    ],

                "Pattern_AnalogFutureMSE":
                    pattern_m[
                        "AnalogFutureMSE"
                    ].to_numpy(),

                "Pattern_RetrievalForecastMSE":
                    pattern_m[
                        "RetrievalForecastMSE"
                    ].to_numpy(),

                "Pattern_NDCG@K":
                    pattern_m[
                        "NDCG@K"
                    ].to_numpy(),

                "Pattern_OracleRecall@K":
                    pattern_m[
                        "OracleRecall@K"
                    ].to_numpy(),

                "Learned_AnalogFutureMSE":
                    learned_mean[
                        "AnalogFutureMSE"
                    ].to_numpy(),

                "Learned_RetrievalForecastMSE":
                    learned_mean[
                        "RetrievalForecastMSE"
                    ].to_numpy(),

                "Learned_NDCG@K":
                    learned_mean[
                        "NDCG@K"
                    ].to_numpy(),

                "Learned_OracleRecall@K":
                    learned_mean[
                        "OracleRecall@K"
                    ].to_numpy(),

                "Learned_AnalogFutureMSE_SeedStd":
                    learned_std[
                        "AnalogFutureMSE"
                    ].to_numpy(),

                "Oracle_AnalogFutureMSE":
                    oracle_m[
                        "OracleAnalogFutureMSE"
                    ].to_numpy(),

                "Oracle_RetrievalForecastMSE":
                    oracle_m[
                        "OracleRetrievalForecastMSE"
                    ].to_numpy(),
            })

            QUERY_RESULTS[
                (
                    dataset_name,
                    H,
                    config
                )
            ] = q

            q.to_parquet(
                RESULT_DIR /
                f"query_level_{dataset_name}_H{H}_{config}.parquet",
                index=False,
            )

            main_rows.append({
                "Dataset":
                    dataset_name,

                "Horizon":
                    H,

                "Config":
                    config,

                "CandidateMode":
                    candidate_mode,

                "TargetMode":
                    target_mode,

                "PatternAnalogFutureMSE":
                    float(
                        pattern_m[
                            "AnalogFutureMSE"
                        ].mean()
                    ),

                "LearnedAnalogFutureMSE":
                    float(
                        learned_mean[
                            "AnalogFutureMSE"
                        ].mean()
                    ),

                "LearnedSeedStd":
                    float(
                        pd.DataFrame(
                            seed_rows
                        )
                        .query(
                            "Dataset == @dataset_name and Horizon == @H and Config == @config"
                        )[
                            "LearnedAnalogFutureMSE"
                        ]
                        .std(
                            ddof=1
                        )
                    ),

                "AnalogImprovement_%":
                    100.0 *
                    (
                        pattern_m[
                            "AnalogFutureMSE"
                        ].mean()
                        -
                        learned_mean[
                            "AnalogFutureMSE"
                        ].mean()
                    )
                    /
                    pattern_m[
                        "AnalogFutureMSE"
                    ].mean(),

                "PatternRetrievalForecastMSE":
                    float(
                        pattern_m[
                            "RetrievalForecastMSE"
                        ].mean()
                    ),

                "LearnedRetrievalForecastMSE":
                    float(
                        learned_mean[
                            "RetrievalForecastMSE"
                        ].mean()
                    ),

                "PatternNDCG@K":
                    float(
                        pattern_m[
                            "NDCG@K"
                        ].mean()
                    ),

                "LearnedNDCG@K":
                    float(
                        learned_mean[
                            "NDCG@K"
                        ].mean()
                    ),

                "PatternOracleRecall@K":
                    float(
                        pattern_m[
                            "OracleRecall@K"
                        ].mean()
                    ),

                "LearnedOracleRecall@K":
                    float(
                        learned_mean[
                            "OracleRecall@K"
                        ].mean()
                    ),

                "PatternOtherChannelFrac@K":
                    float(
                        pattern_m[
                            "OtherChannelFrac@K"
                        ].mean()
                    ),

                "LearnedOtherChannelFrac@K":
                    float(
                        learned_mean[
                            "OtherChannelFrac@K"
                        ].mean()
                    ),

                "OracleWithinM_AnalogFutureMSE":
                    float(
                        oracle_m[
                            "OracleAnalogFutureMSE"
                        ].mean()
                    ),
            })

            oracle_rows.append({
                "Dataset":
                    dataset_name,

                "Horizon":
                    H,

                "Config":
                    config,

                "OracleWithinM_AnalogFutureMSE":
                    float(
                        oracle_m[
                            "OracleAnalogFutureMSE"
                        ].mean()
                    ),

                "OracleWithinM_RetrievalForecastMSE":
                    float(
                        oracle_m[
                            "OracleRetrievalForecastMSE"
                        ].mean()
                    ),
            })

seed_table = pd.DataFrame(
    seed_rows
)

main_table = pd.DataFrame(
    main_rows
)

oracle_table = pd.DataFrame(
    oracle_rows
)

display(
    main_table.sort_values(
        [
            "Dataset",
            "Horizon",
            "TargetMode",
            "LearnedAnalogFutureMSE",
        ]
    )
)

seed_table.to_csv(
    RESULT_DIR /
    "04_seed_results.csv",
    index=False,
)

main_table.to_csv(
    RESULT_DIR /
    "05_main_mechanism_summary.csv",
    index=False,
)

oracle_table.to_csv(
    RESULT_DIR /
    "06_candidate_pool_oracle_summary.csv",
    index=False,
)


## 19. Evaluate Same_Fixed within-channel Shuffled-Future control

In [ ]:

shuffled_rows = []
SHUFFLED_QUERY = {}

for dataset_name in DATASET_NAMES:

    for H in HORIZONS:

        config = "Same_Fixed"

        test_phase = phase_data(
            dataset_name,
            H,
            "test",
            "Same",
            "Fixed",
        )

        frames = []

        for seed in SEEDS:

            model = MODELS[
                (
                    dataset_name,
                    H,
                    config,
                    "Shuffled",
                    seed
                )
            ]

            score = predict_scores(
                model,
                test_phase,
            )

            m = query_metrics(
                score,
                test_phase,
            )

            frames.append(
                m
            )

            shuffled_rows.append({
                "Dataset":
                    dataset_name,

                "Horizon":
                    H,

                "Seed":
                    seed,

                "AnalogFutureMSE":
                    float(
                        m[
                            "AnalogFutureMSE"
                        ].mean()
                    ),

                "RetrievalForecastMSE":
                    float(
                        m[
                            "RetrievalForecastMSE"
                        ].mean()
                    ),

                "NDCG@K":
                    float(
                        m[
                            "NDCG@K"
                        ].mean()
                    ),

                "OracleRecall@K":
                    float(
                        m[
                            "OracleRecall@K"
                        ].mean()
                    ),
            })

        mean_m = pd.DataFrame({
            col:
                np.stack(
                    [
                        f[
                            col
                        ].to_numpy()
                        for f in frames
                    ],
                    axis=0,
                ).mean(
                    axis=0
                )

            for col in frames[
                0
            ].columns
        })

        q = QUERY_RESULTS[
            (
                dataset_name,
                H,
                config
            )
        ].copy()

        q[
            "Shuffled_AnalogFutureMSE"
        ] = mean_m[
            "AnalogFutureMSE"
        ].to_numpy()

        q[
            "Shuffled_RetrievalForecastMSE"
        ] = mean_m[
            "RetrievalForecastMSE"
        ].to_numpy()

        q[
            "Shuffled_NDCG@K"
        ] = mean_m[
            "NDCG@K"
        ].to_numpy()

        q[
            "Shuffled_OracleRecall@K"
        ] = mean_m[
            "OracleRecall@K"
        ].to_numpy()

        SHUFFLED_QUERY[
            (
                dataset_name,
                H
            )
        ] = q

shuffled_seed_table = pd.DataFrame(
    shuffled_rows
)

display(
    shuffled_seed_table
)

shuffled_seed_table.to_csv(
    RESULT_DIR /
    "07_same_fixed_shuffled_seed_results.csv",
    index=False,
)


## 20. Same vs Cross comparison under identical target scaling

In [ ]:

same_cross_rows = []

for dataset_name in DATASET_NAMES:

    for H in HORIZONS:

        for target_mode in [
            "Local",
            "Fixed",
        ]:

            cross = main_table[
                (
                    main_table[
                        "Dataset"
                    ] ==
                    dataset_name
                )
                &
                (
                    main_table[
                        "Horizon"
                    ] ==
                    H
                )
                &
                (
                    main_table[
                        "Config"
                    ] ==
                    f"Cross_{target_mode}"
                )
            ].iloc[
                0
            ]

            same = main_table[
                (
                    main_table[
                        "Dataset"
                    ] ==
                    dataset_name
                )
                &
                (
                    main_table[
                        "Horizon"
                    ] ==
                    H
                )
                &
                (
                    main_table[
                        "Config"
                    ] ==
                    f"Same_{target_mode}"
                )
            ].iloc[
                0
            ]

            same_cross_rows.append({
                "Dataset":
                    dataset_name,

                "Horizon":
                    H,

                "TargetMode":
                    target_mode,

                "CrossPattern":
                    float(
                        cross[
                            "PatternAnalogFutureMSE"
                        ]
                    ),

                "SamePattern":
                    float(
                        same[
                            "PatternAnalogFutureMSE"
                        ]
                    ),

                "CrossLearned":
                    float(
                        cross[
                            "LearnedAnalogFutureMSE"
                        ]
                    ),

                "SameLearned":
                    float(
                        same[
                            "LearnedAnalogFutureMSE"
                        ]
                    ),

                "SameVsCrossLearnedImprovement_%":
                    100.0 *
                    (
                        cross[
                            "LearnedAnalogFutureMSE"
                        ]
                        -
                        same[
                            "LearnedAnalogFutureMSE"
                        ]
                    )
                    /
                    cross[
                        "LearnedAnalogFutureMSE"
                    ],

                "CrossPatternToLearned_%":
                    float(
                        cross[
                            "AnalogImprovement_%"
                        ]
                    ),

                "SamePatternToLearned_%":
                    float(
                        same[
                            "AnalogImprovement_%"
                        ]
                    ),

                "CrossOracleWithinM":
                    float(
                        cross[
                            "OracleWithinM_AnalogFutureMSE"
                        ]
                    ),

                "SameOracleWithinM":
                    float(
                        same[
                            "OracleWithinM_AnalogFutureMSE"
                        ]
                    ),
            })

same_cross_table = pd.DataFrame(
    same_cross_rows
)

display(
    same_cross_table
)

same_cross_table.to_csv(
    RESULT_DIR /
    "08_same_vs_cross_comparison.csv",
    index=False,
)


## 21. Same_Fixed Learned vs Pattern vs Shuffled control

In [ ]:

control_rows = []

for dataset_name in DATASET_NAMES:

    for H in HORIZONS:

        q = SHUFFLED_QUERY[
            (
                dataset_name,
                H
            )
        ]

        pattern = float(
            q[
                "Pattern_AnalogFutureMSE"
            ].mean()
        )

        learned = float(
            q[
                "Learned_AnalogFutureMSE"
            ].mean()
        )

        shuffled = float(
            q[
                "Shuffled_AnalogFutureMSE"
            ].mean()
        )

        control_rows.append({
            "Dataset":
                dataset_name,

            "Horizon":
                H,

            "PatternAnalogFutureMSE":
                pattern,

            "LearnedAnalogFutureMSE":
                learned,

            "ShuffledAnalogFutureMSE":
                shuffled,

            "LearnedVsPattern_%":
                100.0 *
                (
                    pattern -
                    learned
                ) /
                pattern,

            "LearnedVsShuffled_%":
                100.0 *
                (
                    shuffled -
                    learned
                ) /
                shuffled,

            "PatternNDCG@K":
                float(
                    q[
                        "Pattern_NDCG@K"
                    ].mean()
                ),

            "LearnedNDCG@K":
                float(
                    q[
                        "Learned_NDCG@K"
                    ].mean()
                ),

            "ShuffledNDCG@K":
                float(
                    q[
                        "Shuffled_NDCG@K"
                    ].mean()
                ),

            "PatternOracleRecall@K":
                float(
                    q[
                        "Pattern_OracleRecall@K"
                    ].mean()
                ),

            "LearnedOracleRecall@K":
                float(
                    q[
                        "Learned_OracleRecall@K"
                    ].mean()
                ),

            "ShuffledOracleRecall@K":
                float(
                    q[
                        "Shuffled_OracleRecall@K"
                    ].mean()
                ),
        })

control_table = pd.DataFrame(
    control_rows
)

display(
    control_table
)

control_table.to_csv(
    RESULT_DIR /
    "09_same_fixed_shuffled_control.csv",
    index=False,
)


## 22. Moving-block bootstrap

In [ ]:

def moving_block_bootstrap(
    x,
    block_len,
    n_boot,
    seed,
):
    x = np.asarray(
        x,
        dtype=np.float64,
    )

    n = len(
        x
    )

    assert n >= block_len

    rng = np.random.default_rng(
        seed
    )

    n_blocks = math.ceil(
        n /
        block_len
    )

    max_start = (
        n -
        block_len
    )

    boot = np.empty(
        n_boot,
        dtype=np.float64,
    )

    for b in range(
        n_boot
    ):

        parts = []

        for _ in range(
            n_blocks
        ):

            start = rng.integers(
                0,
                max_start +
                1,
            )

            parts.append(
                x[
                    start:
                    start +
                    block_len
                ]
            )

        sample = np.concatenate(
            parts
        )[
            :n
        ]

        boot[
            b
        ] = sample.mean()

    return {
        "ObservedImprovement":
            float(
                x.mean()
            ),

        "CI_2.5%":
            float(
                np.quantile(
                    boot,
                    0.025,
                )
            ),

        "CI_97.5%":
            float(
                np.quantile(
                    boot,
                    0.975,
                )
            ),

        "P_gt_0":
            float(
                (
                    boot >
                    0
                ).mean()
            ),
    }


bootstrap_rows = []

# A. Pattern -> Learned for all four conditions.
for dataset_name in DATASET_NAMES:

    for H in HORIZONS:

        for config in CONFIGS:

            q = QUERY_RESULTS[
                (
                    dataset_name,
                    H,
                    config
                )
            ]

            tmp = pd.DataFrame({
                "Anchor":
                    q[
                        "Anchor"
                    ],

                "Diff":
                    (
                        q[
                            "Pattern_AnalogFutureMSE"
                        ]
                        -
                        q[
                            "Learned_AnalogFutureMSE"
                        ]
                    ),
            })

            anchor_diff = (
                tmp
                .groupby(
                    "Anchor"
                )[
                    "Diff"
                ]
                .mean()
                .sort_index()
                .to_numpy()
            )

            r = moving_block_bootstrap(
                anchor_diff,
                BLOCK_ANCHORS,
                N_BOOT,
                seed=(
                    DATASET_SEED[
                        dataset_name
                    ] +
                    H *
                    100 +
                    CONFIGS.index(
                        config
                    )
                ),
            )

            r.update({
                "Dataset":
                    dataset_name,

                "Horizon":
                    H,

                "Comparison":
                    f"{config}: Pattern -> Learned",

                "SignificantImprovement":
                    bool(
                        r[
                            "CI_2.5%"
                        ] >
                        0
                    ),
            })

            bootstrap_rows.append(
                r
            )

# B. Cross_Fixed -> Same_Fixed Learned.
for dataset_name in DATASET_NAMES:

    for H in HORIZONS:

        cross_q = QUERY_RESULTS[
            (
                dataset_name,
                H,
                "Cross_Fixed"
            )
        ]

        same_q = QUERY_RESULTS[
            (
                dataset_name,
                H,
                "Same_Fixed"
            )
        ]

        # Same query indices/order were preserved.
        np.testing.assert_array_equal(
            cross_q[
                [
                    "Anchor",
                    "ChannelIndex",
                ]
            ].to_numpy(),
            same_q[
                [
                    "Anchor",
                    "ChannelIndex",
                ]
            ].to_numpy(),
        )

        tmp = pd.DataFrame({
            "Anchor":
                same_q[
                    "Anchor"
                ],

            "Diff":
                (
                    cross_q[
                        "Learned_AnalogFutureMSE"
                    ].to_numpy()
                    -
                    same_q[
                        "Learned_AnalogFutureMSE"
                    ].to_numpy()
                ),
        })

        anchor_diff = (
            tmp
            .groupby(
                "Anchor"
            )[
                "Diff"
            ]
            .mean()
            .sort_index()
            .to_numpy()
        )

        r = moving_block_bootstrap(
            anchor_diff,
            BLOCK_ANCHORS,
            N_BOOT,
            seed=(
                50000 +
                DATASET_SEED[
                    dataset_name
                ] +
                H
            ),
        )

        r.update({
            "Dataset":
                dataset_name,

            "Horizon":
                H,

            "Comparison":
                "Fixed: Cross Learned -> Same Learned",

            "SignificantImprovement":
                bool(
                    r[
                        "CI_2.5%"
                    ] >
                    0
                ),
        })

        bootstrap_rows.append(
            r
        )

# C. Same_Fixed Shuffled -> Learned.
for dataset_name in DATASET_NAMES:

    for H in HORIZONS:

        q = SHUFFLED_QUERY[
            (
                dataset_name,
                H
            )
        ]

        tmp = pd.DataFrame({
            "Anchor":
                q[
                    "Anchor"
                ],

            "Diff":
                (
                    q[
                        "Shuffled_AnalogFutureMSE"
                    ]
                    -
                    q[
                        "Learned_AnalogFutureMSE"
                    ]
                ),
        })

        anchor_diff = (
            tmp
            .groupby(
                "Anchor"
            )[
                "Diff"
            ]
            .mean()
            .sort_index()
            .to_numpy()
        )

        r = moving_block_bootstrap(
            anchor_diff,
            BLOCK_ANCHORS,
            N_BOOT,
            seed=(
                60000 +
                DATASET_SEED[
                    dataset_name
                ] +
                H
            ),
        )

        r.update({
            "Dataset":
                dataset_name,

            "Horizon":
                H,

            "Comparison":
                "Same_Fixed: Shuffled -> Learned",

            "SignificantImprovement":
                bool(
                    r[
                        "CI_2.5%"
                    ] >
                    0
                ),
        })

        bootstrap_rows.append(
            r
        )

bootstrap_table = pd.DataFrame(
    bootstrap_rows
)

display(
    bootstrap_table
)

bootstrap_table.to_csv(
    RESULT_DIR /
    "10_moving_block_bootstrap.csv",
    index=False,
)


## 23. Per-channel Weather diagnosis

In [ ]:

weather_channel_rows = []

dataset_name = "Weather"

for H in HORIZONS:

    for config in [
        "Cross_Fixed",
        "Same_Fixed",
    ]:

        q = QUERY_RESULTS[
            (
                dataset_name,
                H,
                config
            )
        ]

        meta = WINDOWS[
            (
                dataset_name,
                H
            )
        ][
            "meta"
        ]

        channel_map = (
            meta[
                [
                    "ChannelIndex",
                    "Channel",
                ]
            ]
            .drop_duplicates()
            .set_index(
                "ChannelIndex"
            )[
                "Channel"
            ]
            .to_dict()
        )

        for cidx, g in q.groupby(
            "ChannelIndex"
        ):

            pattern = float(
                g[
                    "Pattern_AnalogFutureMSE"
                ].mean()
            )

            learned = float(
                g[
                    "Learned_AnalogFutureMSE"
                ].mean()
            )

            weather_channel_rows.append({
                "Horizon":
                    H,

                "Config":
                    config,

                "ChannelIndex":
                    int(
                        cidx
                    ),

                "Channel":
                    str(
                        channel_map[
                            int(
                                cidx
                            )
                        ]
                    ),

                "N":
                    len(
                        g
                    ),

                "PatternAnalogFutureMSE":
                    pattern,

                "LearnedAnalogFutureMSE":
                    learned,

                "Improvement_%":
                    100.0 *
                    (
                        pattern -
                        learned
                    ) /
                    pattern,
            })

weather_channel_table = pd.DataFrame(
    weather_channel_rows
)

display(
    weather_channel_table
)

weather_channel_table.to_csv(
    RESULT_DIR /
    "11_weather_per_channel_analysis.csv",
    index=False,
)


## 24. Mechanism decision

In [ ]:

def get_main(
    dataset,
    H,
    config,
):
    return main_table[
        (
            main_table[
                "Dataset"
            ] ==
            dataset
        )
        &
        (
            main_table[
                "Horizon"
            ] ==
            H
        )
        &
        (
            main_table[
                "Config"
            ] ==
            config
        )
    ].iloc[
        0
    ]


weather_samefixed_pattern_wins = 0
weather_samefixed_cross_wins = 0
weather_samefixed_shuffle_wins = 0
etth1_samefixed_pattern_wins = 0
etth1_samefixed_shuffle_wins = 0

for H in HORIZONS:

    w_sf = get_main(
        "Weather",
        H,
        "Same_Fixed",
    )

    w_cf = get_main(
        "Weather",
        H,
        "Cross_Fixed",
    )

    e_sf = get_main(
        "ETTh1",
        H,
        "Same_Fixed",
    )

    if (
        w_sf[
            "LearnedAnalogFutureMSE"
        ]
        <
        w_sf[
            "PatternAnalogFutureMSE"
        ]
    ):

        weather_samefixed_pattern_wins += 1

    if (
        w_sf[
            "LearnedAnalogFutureMSE"
        ]
        <
        w_cf[
            "LearnedAnalogFutureMSE"
        ]
    ):

        weather_samefixed_cross_wins += 1

    if (
        e_sf[
            "LearnedAnalogFutureMSE"
        ]
        <
        e_sf[
            "PatternAnalogFutureMSE"
        ]
    ):

        etth1_samefixed_pattern_wins += 1

    control_w = control_table[
        (
            control_table[
                "Dataset"
            ] ==
            "Weather"
        )
        &
        (
            control_table[
                "Horizon"
            ] ==
            H
        )
    ].iloc[
        0
    ]

    control_e = control_table[
        (
            control_table[
                "Dataset"
            ] ==
            "ETTh1"
        )
        &
        (
            control_table[
                "Horizon"
            ] ==
            H
        )
    ].iloc[
        0
    ]

    if (
        control_w[
            "LearnedAnalogFutureMSE"
        ]
        <
        control_w[
            "ShuffledAnalogFutureMSE"
        ]
    ):

        weather_samefixed_shuffle_wins += 1

    if (
        control_e[
            "LearnedAnalogFutureMSE"
        ]
        <
        control_e[
            "ShuffledAnalogFutureMSE"
        ]
    ):

        etth1_samefixed_shuffle_wins += 1


weather_cross_fixed_pattern_wins = int(
    (
        main_table[
            (
                main_table[
                    "Dataset"
                ] ==
                "Weather"
            )
            &
            (
                main_table[
                    "Config"
                ] ==
                "Cross_Fixed"
            )
        ][
            "AnalogImprovement_%"
        ] >
        0
    ).sum()
)


if (
    weather_samefixed_pattern_wins == 3
    and
    weather_samefixed_shuffle_wins == 3
    and
    etth1_samefixed_pattern_wins == 3
    and
    etth1_samefixed_shuffle_wins == 3
):

    if weather_samefixed_cross_wins >= 2:

        interpretation = (
            "Mechanism supported: fixed train-scale removes the local-scale pathology, "
            "and semantic same-channel candidate restriction is important for Weather. "
            "Future-compatible supervision remains effective without cross-channel shortcuts."
        )

        next_step = (
            "Stop architecture development. Use Same-channel retrieval for heterogeneous-variable "
            "datasets, retain cross-entity retrieval only where channels are semantically homogeneous, "
            "then proceed to theory and manuscript."
        )

    else:

        interpretation = (
            "Core future-supervision mechanism is robust under fixed train-scale and same-channel retrieval, "
            "but semantic restriction is not the dominant Weather factor. "
            "The main issue was target scaling rather than cross-channel semantics."
        )

        next_step = (
            "Stop architecture development. Formalize the target-scale result and use the empirically "
            "supported candidate policy in the final paper."
        )

elif (
    weather_samefixed_pattern_wins >= 2
    and
    etth1_samefixed_pattern_wins == 3
):

    interpretation = (
        "Partially supported: same-channel fixed-scale retrieval substantially stabilizes Weather, "
        "while ETTh1 remains robust. Weather retains horizon-dependent heterogeneity."
    )

    next_step = (
        "Do not add model complexity. Analyze the remaining failing Weather horizon/channel and "
        "narrow the claim accordingly."
    )

else:

    interpretation = (
        "The Weather failure is not resolved by semantic compatibility and fixed train-scale target."
    )

    next_step = (
        "Do not force a universal cross-domain claim. Keep ETTh1/Electricity/Traffic as supporting domains "
        "and frame Weather as a documented failure case or narrow the paper scope."
    )


decision = pd.DataFrame(
    [
        {
            "Weather_SameFixed_BeatsPattern":
                weather_samefixed_pattern_wins,

            "Weather_SameFixed_BeatsCrossFixed":
                weather_samefixed_cross_wins,

            "Weather_SameFixed_BeatsWithinChannelShuffled":
                weather_samefixed_shuffle_wins,

            "Weather_CrossFixed_BeatsPattern":
                weather_cross_fixed_pattern_wins,

            "ETTh1_SameFixed_BeatsPattern":
                etth1_samefixed_pattern_wins,

            "ETTh1_SameFixed_BeatsWithinChannelShuffled":
                etth1_samefixed_shuffle_wins,

            "Interpretation":
                interpretation,

            "NextStep":
                next_step,
        }
    ]
)

display(
    decision
)

decision.to_csv(
    RESULT_DIR /
    "12_final_mechanism_decision.csv",
    index=False,
)


# Result Reading Guide

Default output directory:

```text
_work/semantic_compatibility/
```

Key files:

```text
01_target_scaling_diagnostics.csv
05_main_mechanism_summary.csv
08_same_vs_cross_comparison.csv
09_same_fixed_shuffled_control.csv
10_moving_block_bootstrap.csv
11_weather_per_channel_analysis.csv
12_final_mechanism_decision.csv
```

## A. Numerical sanity check

Use `01_target_scaling_diagnostics.csv` to verify that the fixed target substantially reduces the extreme tail of Weather target magnitudes.

## B. Same-channel vs. cross-channel

Within the **same fixed target definition**, compare `Cross_Fixed` and `Same_Fixed`. A lower Same_Fixed error indicates that heterogeneous cross-channel candidates were part of the original failure mode.

## C. Future-supervision control

The key ordering is

\[
\boxed{\text{Learned}<\text{ShuffledFutureWithinChannel}}.
\]

Both models use the same channel, candidate pool, network, target marginal distribution, and optimization; they differ only in whether the query is paired with its correct future.

## D. ETTh1 role

If Pattern-to-Learned improvement persists for ETTh1 under Same_Fixed, the effect does not depend on cross-channel candidate mixing.

## E. Candidate-pool ceiling and per-channel analysis

`06_candidate_pool_oracle_summary.csv` and `11_weather_per_channel_analysis.csv` are diagnostics for candidate-pool quality and channel-specific failure modes. They are not used to tune the final architecture.
